# 03 - Model Training & Evaluation

Notebook này train baseline Logistic Regression + model chính Random Forest, phân tích threshold, đánh giá test set, và export artifacts cho backend.

## SECTION 1: Load Processed Data

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import joblib

warnings.filterwarnings('ignore')

# Resolve directories from either project root or notebooks/
processed_candidates = [
    Path('dataset/processed'),
    Path('../dataset/processed'),
]
backend_ml_candidates = [
    Path('backend/ml'),
    Path('../backend/ml'),
]

PROCESSED_DIR = next((p for p in processed_candidates if p.exists()), None)
if PROCESSED_DIR is None:
    raise FileNotFoundError('Cannot find dataset/processed. Please run notebook 02_preprocessing.ipynb first.')

BACKEND_ML_DIR = next((p for p in backend_ml_candidates if p.exists()), None)
if BACKEND_ML_DIR is None:
    raise FileNotFoundError('Cannot find backend/ml directory.')

print('Processed dir:', PROCESSED_DIR.resolve())
print('Backend ml dir:', BACKEND_ML_DIR.resolve())

# Load processed datasets
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_val = pd.read_csv(PROCESSED_DIR / 'X_val.csv')
X_test = pd.read_csv(PROCESSED_DIR / 'X_test.csv')

y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').iloc[:, 0]
y_val = pd.read_csv(PROCESSED_DIR / 'y_val.csv').iloc[:, 0]
y_test = pd.read_csv(PROCESSED_DIR / 'y_test.csv').iloc[:, 0]

# Load and verify feature order
with open(BACKEND_ML_DIR / 'feature_names.json', 'r', encoding='utf-8') as f:
    feature_names = json.load(f)

for split_name, split_df in [('train', X_train), ('val', X_val), ('test', X_test)]:
    missing = [c for c in feature_names if c not in split_df.columns]
    if missing:
        raise ValueError(f'Missing features in X_{split_name}: {missing}')

X_train = X_train[feature_names]
X_val = X_val[feature_names]
X_test = X_test[feature_names]

print('Feature order verified. Number of features:', len(feature_names))
print('X_train:', X_train.shape, 'X_val:', X_val.shape, 'X_test:', X_test.shape)
print('y_train:', y_train.shape, 'y_val:', y_val.shape, 'y_test:', y_test.shape)

## SECTION 2: Baseline - Logistic Regression

In [ ]:
THRESHOLD = 0.3


def evaluate_model(model, X, y, threshold=0.3, title='Model', show_confusion=True):
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    print(f'{title} | threshold={threshold}')
    print(classification_report(y, y_pred, digits=4))

    if show_confusion:
        cm = confusion_matrix(y, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=['On Time', 'Late'])
        disp.plot(cmap='Blues', values_format='d')
        plt.title(f'Confusion Matrix - {title}')
        plt.tight_layout()
        plt.show()

    auc = roc_auc_score(y, y_prob)
    rec_late = recall_score(y, y_pred, pos_label=1)

    print(f'AUC-ROC: {auc:.4f}')
    print(f'{title} Recall (Late class): {rec_late:.4f}')

    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': rec_late,
        'f1': f1_score(y, y_pred, zero_division=0),
        'auc_roc': auc,
    }
    return metrics, y_prob, y_pred


lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

lr_val_metrics, lr_val_prob, lr_val_pred = evaluate_model(
    lr,
    X_val,
    y_val,
    threshold=THRESHOLD,
    title='LR (Validation)'
)

## SECTION 3: Main Model - Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
)
rf.fit(X_train, y_train)

rf_val_metrics, rf_val_prob, rf_val_pred = evaluate_model(
    rf,
    X_val,
    y_val,
    threshold=THRESHOLD,
    title='RF (Validation)'
)

## SECTION 4: Threshold Analysis

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)

rf_val_prob = rf.predict_proba(X_val)[:, 1]
precision_list = []
recall_list = []
f1_list = []

for t in thresholds:
    pred_t = (rf_val_prob >= t).astype(int)
    precision_list.append(precision_score(y_val, pred_t, zero_division=0))
    recall_list.append(recall_score(y_val, pred_t, zero_division=0))
    f1_list.append(f1_score(y_val, pred_t, zero_division=0))

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precision_list, marker='o', label='Precision')
plt.plot(thresholds, recall_list, marker='o', label='Recall')
plt.plot(thresholds, f1_list, marker='o', label='F1')
plt.axvline(x=0.3, color='red', linestyle='--', linewidth=2, label='Threshold = 0.3')
plt.title('Threshold vs Metrics - Random Forest')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## SECTION 5: Feature Importance

In [ ]:
importance_series = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

top_n = min(15, len(importance_series))
top_importance = importance_series.head(top_n)
colors = ['red' if i < 5 else 'steelblue' for i in range(top_n)]

plt.figure(figsize=(10, 6))
plt.barh(top_importance.index[::-1], top_importance.values[::-1], color=colors[::-1])
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print('Top features:')
print(top_importance)

## SECTION 6: Final Evaluation trên Test Set

In [ ]:
# Evaluate LR on test (for comparison table)
lr_test_prob = lr.predict_proba(X_test)[:, 1]
lr_test_pred = (lr_test_prob >= THRESHOLD).astype(int)

lr_test_metrics = {
    'Accuracy': accuracy_score(y_test, lr_test_pred),
    'Recall (Late)': recall_score(y_test, lr_test_pred, zero_division=0),
    'Precision': precision_score(y_test, lr_test_pred, zero_division=0),
    'AUC-ROC': roc_auc_score(y_test, lr_test_prob),
}

# Evaluate RF on test
rf_test_prob = rf.predict_proba(X_test)[:, 1]
rf_test_pred = (rf_test_prob >= THRESHOLD).astype(int)

print('Random Forest - Test classification report (threshold=0.3):')
print(classification_report(y_test, rf_test_pred, digits=4))

rf_test_metrics = {
    'Accuracy': accuracy_score(y_test, rf_test_pred),
    'Recall (Late)': recall_score(y_test, rf_test_pred, zero_division=0),
    'Precision': precision_score(y_test, rf_test_pred, zero_division=0),
    'F1': f1_score(y_test, rf_test_pred, zero_division=0),
    'AUC-ROC': roc_auc_score(y_test, rf_test_prob),
}

cm = confusion_matrix(y_test, rf_test_pred)
ConfusionMatrixDisplay(cm, display_labels=['On Time', 'Late']).plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - RF (Test, threshold=0.3)')
plt.tight_layout()
plt.show()

print('RF Test Metrics:')
for k, v in rf_test_metrics.items():
    print(f'{k}: {v:.4f}')

comparison_df = pd.DataFrame(
    {
        'Metric': ['Accuracy', 'Recall (Late)', 'Precision', 'AUC-ROC'],
        'Logistic Regression': [
            lr_test_metrics['Accuracy'],
            lr_test_metrics['Recall (Late)'],
            lr_test_metrics['Precision'],
            lr_test_metrics['AUC-ROC'],
        ],
        'Random Forest': [
            rf_test_metrics['Accuracy'],
            rf_test_metrics['Recall (Late)'],
            rf_test_metrics['Precision'],
            rf_test_metrics['AUC-ROC'],
        ],
    }
)

print()
print('Comparison Table (Test Set):')
print(comparison_df.to_string(index=False))

## SECTION 7: Export Final Model

In [ ]:
THRESHOLD = 0.3

model_path = BACKEND_ML_DIR / 'model.pkl'
config_path = BACKEND_ML_DIR / 'model_config.json'

joblib.dump(rf, model_path)

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'model_type': 'RandomForestClassifier',
            'threshold': THRESHOLD,
            'n_features': len(feature_names),
            'feature_names': feature_names,
            'target': 'Late_delivery_risk',
            'classes': [0, 1],
            'class_labels': ['On Time', 'Late'],
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f'✅ Model exported: {model_path}')
print(f'✅ Config exported: {config_path}')
print(f'Threshold: {THRESHOLD}')
print(f'Features: {len(feature_names)}')

## SECTION 8: Sanity Check

In [ ]:
loaded_model = joblib.load(BACKEND_ML_DIR / 'model.pkl')

sample = X_test.iloc[:1]
prob = loaded_model.predict_proba(sample)[0][1]
pred = int(prob >= THRESHOLD)

print(f'Sanity check - prob: {prob:.3f}, pred: {pred}')
print('✅ Model loaded and inference OK')